In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
print("=" * 80)
print("LOADING DATA")
print("=" * 80)

train_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print("\nFirst few rows of training data:")
print(train_df.head())

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train_df['SalePrice'], bins=50, edgecolor='black')
axes[0].set_xlabel('Sale Price')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Sale Price')

axes[1].hist(np.log1p(train_df['SalePrice']), bins=50, edgecolor='black')
axes[1].set_xlabel('Log(Sale Price)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Log-Transformed Sale Price')
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("\nMissing Values Analysis:")
train_missing = train_df.isnull().sum()
train_missing = train_missing[train_missing > 0].sort_values(ascending=False)
print(f"\nTop 15 features with missing values in training set:")
print(train_missing.head(15))

# Visualize missing values
if len(train_missing) > 0:
    plt.figure(figsize=(12, 6))
    train_missing.head(20).plot(kind='barh')
    plt.xlabel('Number of Missing Values')
    plt.title('Top 20 Features with Missing Values')
    plt.tight_layout()
    plt.savefig('missing_values.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Correlation analysis
print("\nTop 10 features correlated with SalePrice:")
numeric_features = train_df.select_dtypes(include=[np.number]).columns
correlations = train_df[numeric_features].corr()['SalePrice'].sort_values(ascending=False)
print(correlations.head(11))

# Visualize correlations
plt.figure(figsize=(10, 8))
top_corr_features = correlations.head(11).index[1:]  # Exclude SalePrice itself
sns.heatmap(train_df[top_corr_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Top 10 Features')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
print("\n" + "=" * 80)
print("DATA CLEANING AND FEATURE ENGINEERING")
print("=" * 80)

# Combine train and test for consistent preprocessing
train_len = len(train_df)
y_train = train_df['SalePrice'].copy()
test_ids = test_df['Id'].copy()

# Drop Id and SalePrice from train
train_df = train_df.drop(['Id', 'SalePrice'], axis=1)
test_df = test_df.drop(['Id'], axis=1)

combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
print(f"Combined dataset shape: {combined_df.shape}")


In [ ]:
 # Handle missing values
print("\nHandling missing values...")

# Numerical features - fill with median
numeric_features = combined_df.select_dtypes(include=[np.number]).columns
for col in numeric_features:
    if combined_df[col].isnull().sum() > 0:
        combined_df[col].fillna(combined_df[col].median(), inplace=True)
        
categorical_features = combined_df.select_dtypes(include=['object']).columns
for col in categorical_features:
    if combined_df[col].isnull().sum() > 0:
        combined_df[col].fillna(combined_df[col].mode()[0] if len(combined_df[col].mode()) > 0 else 'None', inplace=True)

print(f"Missing values after cleaning: {combined_df.isnull().sum().sum()}")

In [ ]:
print("\nCreating new features...")

# Total square footage
combined_df['TotalSF'] = combined_df['TotalBsmtSF'] + combined_df['1stFlrSF'] + combined_df['2ndFlrSF']

# Total bathrooms
combined_df['TotalBath'] = combined_df['FullBath'] + 0.5 * combined_df['HalfBath'] + \
                           combined_df['BsmtFullBath'] + 0.5 * combined_df['BsmtHalfBath']

# Age of house
combined_df['HouseAge'] = combined_df['YrSold'] - combined_df['YearBuilt']
combined_df['RemodAge'] = combined_df['YrSold'] - combined_df['YearRemodAdd']

# Total porch area
combined_df['TotalPorchSF'] = combined_df['OpenPorchSF'] + combined_df['EnclosedPorch'] + \
                               combined_df['3SsnPorch'] + combined_df['ScreenPorch']

# Has pool, garage, etc.
combined_df['HasPool'] = (combined_df['PoolArea'] > 0).astype(int)
combined_df['HasGarage'] = (combined_df['GarageArea'] > 0).astype(int)
combined_df['HasBsmt'] = (combined_df['TotalBsmtSF'] > 0).astype(int)
combined_df['HasFireplace'] = (combined_df['Fireplaces'] > 0).astype(int)

print(f"Shape after feature engineering: {combined_df.shape}")

In [ ]:
# Encode categorical variables
print("\nEncoding categorical variables...")
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    combined_df[col] = le.fit_transform(combined_df[col].astype(str))
    label_encoders[col] = le

# Handle outliers in target variable (optional - using log transform instead)
# We'll use log transformation for the target
y_train_log = np.log1p(y_train)

# Split back into train and test
X_train = combined_df[:train_len].copy()
X_test = combined_df[train_len:].copy()

print(f"\nFinal training set shape: {X_train.shape}")
print(f"Final test set shape: {X_test.shape}")

In [ ]:
# Split training data for validation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train_log, test_size=0.2, random_state=42
)

print(f"Training split shape: {X_train_split.shape}")
print(f"Validation split shape: {X_val_split.shape}")

In [ ]:
# ============================================================================
# 4. MODEL TRAINING AND EVALUATION
# ============================================================================
print("\n" + "=" * 80)
print("MODEL TRAINING AND EVALUATION")
print("=" * 80)

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=10),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=15, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, max_depth=5, learning_rate=0.1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, max_depth=5, learning_rate=0.1, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=100, random_state=42, depth=5, learning_rate=0.1, verbose=0),
    'Neural Network': MLPRegressor(hidden_layer_sizes=(100, 50), random_state=42, max_iter=500)
}

results = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train_split, y_train_split)
    
    # Predictions
    y_pred_train = model.predict(X_train_split)
    y_pred_val = model.predict(X_val_split)
    
    # Calculate RMSE (on log scale)
    train_rmse = np.sqrt(mean_squared_error(y_train_split, y_pred_train))
    val_rmse = np.sqrt(mean_squared_error(y_val_split, y_pred_val))
    
    # Calculate R2
    train_r2 = r2_score(y_train_split, y_pred_train)
    val_r2 = r2_score(y_val_split, y_pred_val)
    
    results.append({
        'Model': name,
        'Train RMSE': train_rmse,
        'Validation RMSE': val_rmse,
        'Train R2': train_r2,
        'Validation R2': val_r2
    })
    
    print(f"  Train RMSE: {train_rmse:.4f}")
    print(f"  Validation RMSE: {val_rmse:.4f}")
    print(f"  Train R2: {train_r2:.4f}")
    print(f"  Validation R2: {val_r2:.4f}")




In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Validation RMSE')

print("\n" + "=" * 80)
print("MODEL COMPARISON RESULTS")
print("=" * 80)
print(results_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

results_df_plot = results_df.sort_values('Validation RMSE', ascending=False)

axes[0].barh(results_df_plot['Model'], results_df_plot['Validation RMSE'], color='steelblue')
axes[0].set_xlabel('Validation RMSE (Log Scale)')
axes[0].set_title('Model Comparison - RMSE')
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(results_df_plot['Model'], results_df_plot['Validation R2'], color='coral')
axes[1].set_xlabel('Validation R² Score')
axes[1].set_title('Model Comparison - R² Score')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# 5. BEST MODEL AND FEATURE IMPORTANCE
# ============================================================================
print("\n" + "=" * 80)
print("BEST MODEL ANALYSIS")
print("=" * 80)

best_model_name = results_df.iloc[0]['Model']
print(f"\nBest Model: {best_model_name}")
print(f"Validation RMSE: {results_df.iloc[0]['Validation RMSE']:.4f}")

# Retrain best model on full training data
best_model = models[best_model_name]
best_model.fit(X_train, y_train_log)

# Feature importance (if available)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    print(feature_importance.head(10).to_string(index=False))
    
    # Visualize top 10 features
    plt.figure(figsize=(10, 6))
    top_features = feature_importance.head(10)
    plt.barh(top_features['Feature'], top_features['Importance'], color='forestgreen')
    plt.xlabel('Feature Importance')
    plt.title(f'Top 10 Most Important Features - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================================
# 6. GENERATE PREDICTIONS AND SUBMISSION
# ============================================================================
print("\n" + "=" * 80)
print("GENERATING PREDICTIONS")
print("=" * 80)

# Make predictions on test set
y_pred_test_log = best_model.predict(X_test)
y_pred_test = np.expm1(y_pred_test_log)  # Convert back from log scale

# Create submission file
submission = pd.DataFrame({
    'Id': test_ids.astype(int),
    'SalePrice': y_pred_test  # Keep as float with decimals
})

# Verify data types
print("=== SUBMISSION DATA TYPES ===")
print(submission.dtypes)
print(f"Id type: {submission['Id'].dtype}")
print(f"SalePrice type: {submission['SalePrice'].dtype}")

# Save with float format (NOT integers)
submission.to_csv('submission.csv', index=False, float_format='%.10f')

print("\n=== SUBMISSION FILE CREATED ===")
print("File saved as: submission.csv")
print(f"Number of predictions: {len(submission)}")

print("\nFirst few predictions:")
print(submission.head(10))

print("\nLast few predictions:")
print(submission.tail(5))

print("\n=== PREDICTION STATISTICS ===")
print(submission['SalePrice'].describe())
print(f"\nMinimum predicted price: ${submission['SalePrice'].min():,.2f}")
print(f"Maximum predicted price: ${submission['SalePrice'].max():,.2f}")
print(f"Mean predicted price: ${submission['SalePrice'].mean():,.2f}")
print(f"Median predicted price: ${submission['SalePrice'].median():,.2f}")

# Verify no missing values
print(f"\nMissing values in submission: {submission.isnull().sum().sum()}")
print(f"Total rows in submission: {len(submission)}")
print(f"Expected rows (test set): {len(test_ids)}")

# Check for any anomalies
print("\n=== VALIDATION CHECKS ===")
if (submission['SalePrice'] < 0).any():
    print("⚠️ WARNING: Negative prices detected!")
else:
    print("✓ All prices are positive")

if submission['Id'].duplicated().any():
    print("⚠️ WARNING: Duplicate IDs detected!")
else:
    print("✓ All IDs are unique")

if len(submission) == len(test_ids):
    print(f"✓ Correct number of predictions: {len(submission)}")
else:
    print(f"⚠️ WARNING: Expected {len(test_ids)} predictions, got {len(submission)}")

print("\n✅ Submission file is ready for Kaggle upload!")